In [ ]:
# Import required libraries

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score

from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import roc_curve
from sklearn.metrics import roc_auc_score

In [ ]:
# Load cleaned dataset

df = pd.read_csv("cleaned_data.csv")

print(df.shape)

df.head()

In [ ]:
# Select features and target columns

X = df.drop(columns=["Fare", "Survived"])

# Regression target
y_reg = df["Fare"]

# Classification target
y_clf = df["Survived"]

In [ ]:
# Check feature data types

print(X.dtypes)

In [ ]:
# Convert categorical columns into numeric values

# Drop unnecessary high-cardinality text columns to prevent overfitting
X = df.drop(columns=['PassengerId', 'Name', 'Ticket', 'Cabin', 'Fare', 'Survived'], errors='ignore')

# Convert remaining categorical columns (like Sex, Embarked) into dummy/one-hot variables
X = pd.get_dummies(X, drop_first=True)

# View the cleaned features and check the new shape
print(X.head())
print("\nX shape after fix:", X.shape)

In [ ]:
# Train test split for regression

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X,
    y_reg,
    test_size=0.2,
    random_state=42
)

In [ ]:
# Train test split for classification

X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X,
    y_clf,
    test_size=0.2,
    random_state=42
)

In [ ]:
# Scale regression data

scaler_reg = StandardScaler()

X_train_reg = scaler_reg.fit_transform(X_train_reg)
X_test_reg = scaler_reg.transform(X_test_reg)


# Scale classification data

scaler_clf = StandardScaler()

X_train_clf = scaler_clf.fit_transform(X_train_clf)
X_test_clf = scaler_clf.transform(X_test_clf)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

linear_model = LinearRegression()

linear_model.fit(X_train_reg, y_train_reg)

y_pred_reg = linear_model.predict(X_test_reg)

mse = mean_squared_error(y_test_reg, y_pred_reg)
r2 = r2_score(y_test_reg, y_pred_reg)

print("Mean Squared Error:", mse)
print("R2 Score:", r2)

In [ ]:
# Display model coefficients

coef_df = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": linear_model.coef_
})

print(coef_df)

In [ ]:
# Display model coefficients

coef_df = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": linear_model.coef_
})

print(coef_df)

In [ ]:
# find top 3 important features

coef_df["Absolute"] = coef_df["Coefficient"].abs()

top_features = coef_df.sort_values(by="Absolute", ascending=False)

print(top_features.head(3))

In [ ]:
# train Ridge Regression model

ridge_model = Ridge(alpha=1.0)

ridge_model.fit(X_train_reg, y_train_reg)

In [ ]:
# predict using Ridge Regression

ridge_pred = ridge_model.predict(X_test_reg)

ridge_mse = mean_squared_error(y_test_reg, ridge_pred)

ridge_r2 = r2_score(y_test_reg, ridge_pred)

print("Ridge Mean Squared Error:", ridge_mse)
print("Ridge R2 Score:", ridge_r2)

In [ ]:
# compare both regression models

comparison = pd.DataFrame({
    "Model": ["Linear Regression", "Ridge Regression"],
    "MSE": [mse, ridge_mse],
    "R2 Score": [r2, ridge_r2]
})

print(comparison)

In [ ]:
# check class distribution
print("Class Distribution")
print(y_train_clf.value_counts())
print("\nClass Percentage")
print(round(y_train_clf.value_counts(normalize=True) * 100, 2))

In [ ]:
# train Logistic Regression model

log_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)

log_model.fit(X_train_clf, y_train_clf)

In [ ]:
# predict class labels,probabilities

y_pred = log_model.predict(X_test_clf)

y_prob = log_model.predict_proba(X_test_clf)[:, 1]

In [ ]:
# evaluate classification model

cm = confusion_matrix(y_test_clf, y_pred)

print("Confusion Matrix")
print(cm)

print("\nClassification Report")
print(classification_report(y_test_clf, y_pred))

print("Accuracy :", accuracy_score(y_test_clf, y_pred))
print("Precision :", precision_score(y_test_clf, y_pred))
print("Recall :", recall_score(y_test_clf, y_pred))
print("F1 Score :", f1_score(y_test_clf, y_pred))

In [ ]:
# plot ROC Curve
fpr, tpr, thresholds = roc_curve(y_test_clf, y_prob)
auc = roc_auc_score(y_test_clf, y_prob)
plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f"AUC = {auc:.3f}")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()
print("AUC Score:", auc)

In [ ]:
# check model performance at different thresholds

thresholds = [0.30, 0.40, 0.50, 0.60, 0.70]
results = []

for t in thresholds:

    pred = (y_prob >= t).astype(int)

    results.append([
        t,
        precision_score(y_test_clf, pred),
        recall_score(y_test_clf, pred),
        f1_score(y_test_clf, pred)
    ])

threshold_df = pd.DataFrame(
    results,
    columns=["Threshold", "Precision", "Recall", "F1 Score"]
)
print(threshold_df)

In [ ]:
# train logistic regression with strong regularization

log_model_c01 = LogisticRegression(
    C=0.01,
    max_iter=1000,
    class_weight="balanced"
)

log_model_c01.fit(X_train_clf, y_train_clf)

y_pred_c01 = log_model_c01.predict(X_test_clf)

y_prob_c01 = log_model_c01.predict_proba(X_test_clf)[:, 1]

In [ ]:
# compare both Logistic Regression models

auc_c01 = roc_auc_score(y_test_clf, y_prob_c01)

comparison_df = pd.DataFrame({
    "Model": [
        "Logistic Regression (C=1.0)",
        "Logistic Regression (C=0.01)"
    ],
    "Precision": [
        precision_score(y_test_clf, y_pred),
        precision_score(y_test_clf, y_pred_c01)
    ],
    "Recall": [
        recall_score(y_test_clf, y_pred),
        recall_score(y_test_clf, y_pred_c01)
    ],
    "AUC": [
        auc,
        auc_c01
    ]
})

print(comparison_df)

In [ ]:
# bootstrap sampling

auc_difference = []

for i in range(500):

    sample_index = np.random.choice(
        len(y_test_clf),
        size=len(y_test_clf),
        replace=True
    )

    sample_y = y_test_clf.iloc[sample_index]

    sample_prob1 = y_prob[sample_index]

    sample_prob2 = y_prob_c01[sample_index]

    auc1 = roc_auc_score(sample_y, sample_prob1)

    auc2 = roc_auc_score(sample_y, sample_prob2)

    auc_difference.append(auc1 - auc2)

In [ ]:
# bootstrap results

mean_difference = np.mean(auc_difference)

lower_limit = np.percentile(auc_difference, 2.5)

upper_limit = np.percentile(auc_difference, 97.5)

print("Mean AUC Difference :", mean_difference)

print("95% Confidence Interval")

print("Lower Limit :", lower_limit)

print("Upper Limit :", upper_limit)